#  DINOv2 + YOLO Bird Classification Pipeline


## Step 1: Setup and Installations

In [ ]:
# Install required packages
!pip install ultralytics transformers torch torchvision scikit-learn pandas numpy pillow opencv-python tqdm -q

print("All packages installed!")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("Google Drive mounted!")

In [ ]:
# Imports
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoImageProcessor, Dinov2Model
from ultralytics import YOLO
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import json
from datetime import datetime
import cv2
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("All imports successful!")

## Step 2: Configuration

In [ ]:
# Paths
BASE_PATH = Path("/content/drive/MyDrive/bird_classification/bdma-07-competition-2026/BDMA7_project_files")
TRAIN_IMG_DIR = BASE_PATH / "train_images"
VAL_IMG_DIR = BASE_PATH / "val_images"
TEST_IMG_DIR = BASE_PATH / "test_images/mistery_cat"
OUTPUT_DIR = BASE_PATH / "dinov2_results"
OUTPUT_DIR.mkdir(exist_ok=True)

CROPPED_DIR = OUTPUT_DIR / "cropped_images"
CROPPED_DIR.mkdir(exist_ok=True)

# Training configuration
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
EPOCHS = 20
N_FOLDS = 3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("="*80)
print("🦅 DINOv2 + YOLO Bird Classification Pipeline")
print("="*80)
print(f"Device: {DEVICE}")
print(f"Data directory exists: {BASE_PATH.exists()}")
print(f"Output directory: {OUTPUT_DIR}")

## Step 3: YOLO Bird Detection & Cropping

This step detects birds in images and crops them with a margin. This focuses the model on the bird rather than background.

In [ ]:
def crop_bird_from_image(image_path, output_path, detector, margin=0.2):
    """
    Detect bird in image and crop to bounding box with margin

    Args:
        image_path: Path to input image
        output_path: Path to save cropped image
        detector: YOLO detector
        margin: Margin to add around bounding box (0.2 = 20%)

    Returns:
        bool: True if successful, False otherwise
    """
    try:
        # Read image
        img = cv2.imread(str(image_path))
        if img is None:
            return False

        h, w = img.shape[:2]

        # Detect birds (class 14 in COCO dataset)
        results = detector(image_path, classes=[14], verbose=False)

        if len(results[0].boxes) == 0:
            # No bird detected - save original with resize
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(img_rgb)
            pil_img = pil_img.resize((224, 224))
            pil_img.save(output_path)
            return True

        # Get largest detection (assume it's the main bird)
        boxes = results[0].boxes.xyxy.cpu().numpy()
        areas = [(box[2] - box[0]) * (box[3] - box[1]) for box in boxes]
        largest_idx = np.argmax(areas)
        box = boxes[largest_idx]

        x1, y1, x2, y2 = box

        # Add margin
        box_w = x2 - x1
        box_h = y2 - y1

        x1 = max(0, x1 - box_w * margin)
        y1 = max(0, y1 - box_h * margin)
        x2 = min(w, x2 + box_w * margin)
        y2 = min(h, y2 + box_h * margin)

        # Crop
        cropped = img[int(y1):int(y2), int(x1):int(x2)]

        # Convert to RGB and save
        cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(cropped_rgb)
        pil_img.save(output_path)

        return True

    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return False

print("Cropping function defined")

In [ ]:
# Load YOLO detector
print("Loading YOLO detector...")
detector = YOLO('yolov8x.pt')  # Using largest model for best accuracy
print("YOLO loaded")

In [ ]:
# Process training images
print("\n" + "="*80)
print("Processing Training Images")
print("="*80)

train_cropped_dir = CROPPED_DIR / "train"
train_cropped_dir.mkdir(exist_ok=True)

# Iterate through class folders
class_folders = [d for d in TRAIN_IMG_DIR.iterdir() if d.is_dir()]
print(f"Found {len(class_folders)} class folders in training set")

total_train_images = 0
for class_folder in tqdm(class_folders, desc="Processing classes"):
    class_name = class_folder.name

    # Create corresponding output directory
    output_class_dir = train_cropped_dir / class_name
    output_class_dir.mkdir(exist_ok=True)

    # Get all images in this class
    class_images = list(class_folder.glob('*.jpg')) + list(class_folder.glob('*.png'))
    total_train_images += len(class_images)

    # Crop each image
    for img_path in class_images:
        output_path = output_class_dir / img_path.name
        if not output_path.exists():
            crop_bird_from_image(img_path, output_path, detector)

print(f"\n {total_train_images} training images cropped and saved to {train_cropped_dir}")

In [ ]:
# Process validation images
print("\n" + "="*80)
print("Processing Validation Images")
print("="*80)

val_cropped_dir = CROPPED_DIR / "val"
val_cropped_dir.mkdir(exist_ok=True)

# Iterate through class folders
class_folders = [d for d in VAL_IMG_DIR.iterdir() if d.is_dir()]
print(f"Found {len(class_folders)} class folders in validation set")

total_val_images = 0
for class_folder in tqdm(class_folders, desc="Processing classes"):
    class_name = class_folder.name

    # Create corresponding output directory
    output_class_dir = val_cropped_dir / class_name
    output_class_dir.mkdir(exist_ok=True)

    # Get all images in this class
    class_images = list(class_folder.glob('*.jpg')) + list(class_folder.glob('*.png'))
    total_val_images += len(class_images)

    # Crop each image
    for img_path in class_images:
        output_path = output_class_dir / img_path.name
        if not output_path.exists():
            crop_bird_from_image(img_path, output_path, detector)

print(f"\n {total_val_images} validation images cropped and saved to {val_cropped_dir}")

In [ ]:
# Process test images
print("\n" + "="*80)
print("Processing Test Images")
print("="*80)

test_cropped_dir = CROPPED_DIR / "test"
test_cropped_dir.mkdir(exist_ok=True)

test_images = list(TEST_IMG_DIR.glob('*.jpg')) + list(TEST_IMG_DIR.glob('*.png'))
print(f"Found {len(test_images)} test images")

for img_path in tqdm(test_images, desc="Cropping test images"):
    output_path = test_cropped_dir / img_path.name
    if not output_path.exists():
        crop_bird_from_image(img_path, output_path, detector)

print(f" Test images cropped and saved to {test_cropped_dir}")

## Step 4: Load Data from Directory Structure

In [ ]:
# Build dataset from directory structure
# Structure: train_images/species_name/*.jpg
#           val_images/species_name/*.jpg

def load_images_from_directory(base_dir, cropped_dir):
    """
    Load images from directory structure where each class is a folder

    Args:
        base_dir: Original images directory
        cropped_dir: Cropped images directory (mirrors structure)

    Returns:
        DataFrame with columns: original_path, cropped_path, class_name
    """
    data = []

    # Iterate through class folders
    class_folders = [d for d in base_dir.iterdir() if d.is_dir()]

    for class_folder in class_folders:
        class_name = class_folder.name

        # Get all images in this class folder
        image_files = list(class_folder.glob('*.jpg')) + list(class_folder.glob('*.png'))

        for img_path in image_files:
            # Cropped image path mirrors the structure
            relative_path = img_path.relative_to(base_dir)
            cropped_path = cropped_dir / relative_path

            data.append({
                'original_path': str(img_path),
                'cropped_path': str(cropped_path),
                'class_name': class_name,
                'image_name': img_path.name
            })

    return pd.DataFrame(data)

print("Loading data from directory structure...")

# Load train and val data
train_df = load_images_from_directory(TRAIN_IMG_DIR, train_cropped_dir)
val_df = load_images_from_directory(VAL_IMG_DIR, val_cropped_dir)

print(f"Train images: {len(train_df)}")
print(f"Val images: {len(val_df)}")

# Combine into single dataset
combined_df = pd.concat([train_df, val_df], ignore_index=True)

# Create class mapping (alphabetical for consistency)
unique_classes = sorted(combined_df['class_name'].unique())
class_mapping = {name: idx for idx, name in enumerate(unique_classes)}
combined_df['label'] = combined_df['class_name'].map(class_mapping).astype('int64')

# Use cropped_path as full_path for training
combined_df['full_path'] = combined_df['cropped_path']

print(f"\nTotal samples: {len(combined_df)}")
print(f"Number of classes: {len(class_mapping)}")
print(f"Samples per class: ~{len(combined_df) / len(class_mapping):.0f}")

# Show class distribution
print("\nClass distribution:")
class_counts = combined_df['class_name'].value_counts().sort_index()
for class_name, count in class_counts.items():
    print(f"  {class_mapping[class_name]:2d}. {class_name:30s}: {count:3d} images")

# Save class mapping
with open(OUTPUT_DIR / 'class_mapping.json', 'w') as f:
    json.dump(class_mapping, f, indent=2)

print(f"\n Class mapping saved to {OUTPUT_DIR / 'class_mapping.json'}")

## Step 5: Setup DINOv2 Model

In [ ]:
# Load DINOv2 processor and model
print("Loading facebook/dinov2-base...")
processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
dinov2_backbone = Dinov2Model.from_pretrained('facebook/dinov2-base')

# Freeze the backbone
for param in dinov2_backbone.parameters():
    param.requires_grad = False

dinov2_backbone = dinov2_backbone.to(DEVICE)

print(" DINOv2 loaded and frozen")
print(f"  Embedding dimension: 768")
print(f"  Parameters frozen: {sum(p.numel() for p in dinov2_backbone.parameters()):,}")

In [ ]:
# Define transforms
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

print(" Transforms defined")

In [ ]:
# Dataset class
class BirdDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['full_path']).convert('RGB')
        label = row['label']

        if self.transform:
            image = self.transform(image)

        return image, label

print(" Dataset class defined")

In [ ]:
# DINOv2 Classifier Model
class DINOv2Classifier(nn.Module):
    """
    DINOv2 with frozen backbone + trainable classifier head
    """
    def __init__(self, backbone, num_classes, hidden_dim=512, dropout=0.3):
        super().__init__()
        self.backbone = backbone

        # Classification head with more capacity for fine-grained features
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(dropout),
            nn.Linear(768, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # Extract features from frozen DINOv2
        with torch.no_grad():
            outputs = self.backbone(x)
            # Use CLS token (first token)
            features = outputs.last_hidden_state[:, 0]

        # Classify
        logits = self.classifier(features)
        return logits

print(" DINOv2Classifier defined")

## Step 6: Training Function

In [ ]:
def train_dinov2_classifier(train_loader, val_loader, fold_idx):
    """Train DINOv2 classifier with frozen backbone"""

    print(f"\n{'='*60}")
    print(f"TRAINING FOLD {fold_idx + 1}")
    print(f"{'='*60}")

    # Create model
    model = DINOv2Classifier(
        backbone=dinov2_backbone,
        num_classes=len(class_mapping),
        hidden_dim=512,
        dropout=0.3
    )
    model = model.to(DEVICE)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(
        model.classifier.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4
    )

    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_val_acc = 0.0
    best_model_state = None
    patience = 5
    patience_counter = 0

    for epoch in range(EPOCHS):
        # Training
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]', leave=False)
        for images, labels in pbar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()

            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100.*train_correct/train_total:.2f}%'
            })

        train_acc = 100. * train_correct / train_total
        avg_train_loss = train_loss / len(train_loader)

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]', leave=False):
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        val_acc = 100. * val_correct / val_total

        print(f"Epoch {epoch+1:2d}: Train Loss: {avg_train_loss:.4f} | "
              f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            patience_counter = 0
            print(f" New best validation accuracy!")
        else:
            patience_counter += 1

        # Early stopping
        if patience_counter >= patience:
            print(f"  Early stopping triggered after {epoch+1} epochs")
            break

        scheduler.step()

    print(f"\nFold {fold_idx + 1} Best Val Acc: {best_val_acc:.2f}%")

    return {
        'model_state': best_model_state,
        'best_val_acc': best_val_acc
    }

print(" Training function defined")

## Step 7: K-Fold Training

In [ ]:
print("="*80)
print(" STARTING K-FOLD CROSS-VALIDATION")
print("="*80)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_results = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(combined_df, combined_df['label'])):
    # Create datasets
    train_fold_df = combined_df.iloc[train_idx]
    val_fold_df = combined_df.iloc[val_idx]

    train_dataset = BirdDataset(train_fold_df, transform=train_transform)
    val_dataset = BirdDataset(val_fold_df, transform=val_transform)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    print(f"\nFold {fold_idx + 1}: Train={len(train_dataset)}, Val={len(val_dataset)}")

    # Train model
    result = train_dinov2_classifier(train_loader, val_loader, fold_idx)
    fold_results.append(result)

print("\n All folds trained!")

## Step 8: Save Models

In [ ]:
# Save models
model_save_dir = OUTPUT_DIR / 'dinov2_models'
model_save_dir.mkdir(exist_ok=True)

for fold_idx, result in enumerate(fold_results):
    save_path = model_save_dir / f'fold_{fold_idx}.pth'
    torch.save(result['model_state'], save_path)
    print(f"Saved fold {fold_idx} to {save_path}")

print(f"\n All models saved to {model_save_dir}")

## Step 9: Training Summary

In [ ]:
print("="*80)
print(" TRAINING SUMMARY")
print("="*80)

val_accs = [r['best_val_acc'] for r in fold_results]
mean_val_acc = np.mean(val_accs)
std_val_acc = np.std(val_accs)

print(f"\nValidation Accuracies:")
for fold_idx, acc in enumerate(val_accs):
    print(f"  Fold {fold_idx + 1}: {acc:.2f}%")

print(f"\nMean: {mean_val_acc:.2f}% ± {std_val_acc:.2f}%")

# Save metadata
metadata = {
    'timestamp': datetime.now().isoformat(),
    'model': 'dinov2-base + classifier',
    'strategy': 'yolo_crop + frozen_backbone',
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'batch_size': BATCH_SIZE,
    'mean_val_acc': mean_val_acc,
    'std_val_acc': std_val_acc,
    'fold_accs': val_accs,
    'class_mapping': class_mapping
}

with open(OUTPUT_DIR / 'training_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n Training complete!")
print(f" Models saved to {model_save_dir}")
print(f" Expected test accuracy: {mean_val_acc:.1f}% ± {std_val_acc:.1f}%")

## Step 10: Evaluation on Train/Val Set

In [ ]:
print("="*80)
print(" EVALUATING ON TRAIN/VAL SET")
print("="*80)

eval_dataset = BirdDataset(combined_df, transform=val_transform)

all_fold_preds = []
all_fold_labels = []

for fold in range(N_FOLDS):
    print(f"\nFold {fold + 1}/{N_FOLDS}")

    # Load model
    model = DINOv2Classifier(
        backbone=dinov2_backbone,
        num_classes=len(class_mapping),
        hidden_dim=512,
        dropout=0.3
    )

    model_path = model_save_dir / f'fold_{fold}.pth'
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model = model.to(DEVICE)
    model.eval()

    eval_loader = DataLoader(
        eval_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    fold_preds = []
    fold_labels = []

    with torch.no_grad():
        for images, labels in tqdm(eval_loader, desc=f'Fold {fold+1} Eval'):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)

            fold_preds.extend(predictions.cpu().numpy())
            fold_labels.extend(labels.cpu().numpy())

    fold_preds = np.array(fold_preds)
    fold_labels = np.array(fold_labels)

    all_fold_preds.append(fold_preds)
    all_fold_labels.append(fold_labels)

    accuracy = (fold_preds == fold_labels).mean()
    print(f"Fold {fold + 1} Accuracy: {accuracy:.4f}")

# Ensemble
all_fold_preds = np.array(all_fold_preds)
all_labels = all_fold_labels[0]

ensemble_preds, _ = stats.mode(all_fold_preds, axis=0, keepdims=False)

print(f"\n{'='*80}")
print("ENSEMBLE RESULTS")
print(f"{'='*80}")
print(f"Ensemble Accuracy: {(ensemble_preds == all_labels).mean():.4f}")

## Step 11: Generate Predictions for Mystery Test Set

In [ ]:
print("="*80)
print(" GENERATING PREDICTIONS FOR MYSTERY TEST SET")
print("="*80)

# Create test dataset
class TestDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, str(img_path.name)

# Load cropped test images
test_images = sorted(list(test_cropped_dir.glob('*.jpg')))
print(f"Found {len(test_images)} test images")

test_dataset = TestDataset(test_images, transform=val_transform)

# Generate predictions from all folds
all_fold_predictions = []

for fold in range(N_FOLDS):
    print(f"\nFold {fold + 1}/{N_FOLDS}")

    model = DINOv2Classifier(
        backbone=dinov2_backbone,
        num_classes=len(class_mapping),
        hidden_dim=512,
        dropout=0.3
    )

    model_path = model_save_dir / f'fold_{fold}.pth'
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model = model.to(DEVICE)
    model.eval()

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    fold_preds = []
    filenames = []

    with torch.no_grad():
        for images, names in tqdm(test_loader, desc=f'Fold {fold+1} Predictions'):
            images = images.to(DEVICE)

            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)

            fold_preds.extend(predictions.cpu().numpy())
            filenames.extend(names)

    all_fold_predictions.append(np.array(fold_preds))

# Ensemble predictions
all_fold_predictions = np.array(all_fold_predictions)
ensemble_predictions, _ = stats.mode(all_fold_predictions, axis=0, keepdims=False)

print(f"\nGenerated {len(ensemble_predictions)} predictions")

## Step 11.5: Test-Time Augmentation (TTA) & Enhanced Ensemble

Apply TTA to improve predictions with multiple augmented views per image.

In [ ]:
# TTA Transforms
import torchvision.transforms.functional as TF

class TTATransforms:
    """Test-Time Augmentation transforms"""

    def __init__(self, base_transform):
        self.base_transform = base_transform

    def get_tta_transforms(self):
        """Returns list of TTA transform functions"""
        return [
            self._original,
            self._hflip,
            self._brightness_up,
            self._brightness_down,
            self._contrast_up,
        ]

    def _original(self, image):
        return self.base_transform(image)

    def _hflip(self, image):
        image = TF.hflip(image)
        return self.base_transform(image)

    def _brightness_up(self, image):
        image = TF.adjust_brightness(image, brightness_factor=1.2)
        return self.base_transform(image)

    def _brightness_down(self, image):
        image = TF.adjust_brightness(image, brightness_factor=0.8)
        return self.base_transform(image)

    def _contrast_up(self, image):
        image = TF.adjust_contrast(image, contrast_factor=1.2)
        return self.base_transform(image)

print(" TTA transforms defined")

In [ ]:
print("="*80)
print(" GENERATING TTA PREDICTIONS (YOLO CROPS)")
print("="*80)

# Initialize TTA
tta = TTATransforms(val_transform)
tta_transforms = tta.get_tta_transforms()

# Storage for all predictions
all_tta_predictions = []  # Shape: (n_folds, n_tta, n_samples)

for fold in range(N_FOLDS):
    print(f"\nFold {fold + 1}/{N_FOLDS}")

    # Load model
    model = DINOv2Classifier(
        backbone=dinov2_backbone,
        num_classes=len(class_mapping),
        hidden_dim=512,
        dropout=0.3
    )
    model_path = model_save_dir / f'fold_{fold}.pth'
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model = model.to(DEVICE)
    model.eval()

    fold_tta_preds = []  # Shape: (n_tta, n_samples)

    # For each TTA transform
    for tta_idx, tta_transform in enumerate(tta_transforms):
        print(f"  TTA variant {tta_idx + 1}/{len(tta_transforms)}", end='\r')

        # Create dataset with this TTA transform
        class TTATestDataset(Dataset):
            def __init__(self, image_paths, transform):
                self.image_paths = image_paths
                self.transform = transform

            def __len__(self):
                return len(self.image_paths)

            def __getitem__(self, idx):
                img_path = self.image_paths[idx]
                image = Image.open(img_path).convert('RGB')
                image = self.transform(image)
                return image, str(img_path.name)

        tta_dataset = TTATestDataset(test_images, tta_transform)
        tta_loader = DataLoader(
            tta_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )

        # Get predictions
        tta_preds = []
        tta_logits = []

        with torch.no_grad():
            for images, names in tta_loader:
                images = images.to(DEVICE)
                outputs = model(images)

                # Store both predictions and logits
                tta_logits.extend(outputs.cpu().numpy())
                predictions = torch.argmax(outputs, dim=1)
                tta_preds.extend(predictions.cpu().numpy())

        fold_tta_preds.append(np.array(tta_preds))

    print(f"  Completed {len(tta_transforms)} TTA variants")
    all_tta_predictions.append(np.array(fold_tta_preds))

all_tta_predictions = np.array(all_tta_predictions)  # Shape: (n_folds, n_tta, n_samples)
print(f"\n TTA predictions shape: {all_tta_predictions.shape}")
print(f"  (folds={N_FOLDS}, tta_variants={len(tta_transforms)}, samples={len(test_images)})")

In [ ]:
print("="*80)
print("  TRAINING COMPLEMENTARY MODEL ON FULL IMAGES (NO YOLO CROP)")
print("="*80)

# Create datasets from ORIGINAL images (not cropped)
train_original_dir = BASE_PATH / 'train_images'
val_original_dir = BASE_PATH / 'val_images'

# Build dataframes for original images
train_full_data = []
for species_dir in train_original_dir.iterdir():
    if species_dir.is_dir():
        for img_path in species_dir.glob('*.jpg'):
            train_full_data.append({
                'full_path': str(img_path),
                'class_name': species_dir.name,
                'label': class_mapping[species_dir.name]
            })

val_full_data = []
for species_dir in val_original_dir.iterdir():
    if species_dir.is_dir():
        for img_path in species_dir.glob('*.jpg'):
            val_full_data.append({
                'full_path': str(img_path),
                'class_name': species_dir.name,
                'label': class_mapping[species_dir.name]
            })

train_full_df = pd.DataFrame(train_full_data)
val_full_df = pd.DataFrame(val_full_data)

print(f"Train (full images): {len(train_full_df)} images")
print(f"Val (full images): {len(val_full_df)} images")

# Train single model on full images
train_full_dataset = BirdDataset(train_full_df, transform=train_transform)
val_full_dataset = BirdDataset(val_full_df, transform=val_transform)

train_full_loader = DataLoader(
    train_full_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_full_loader = DataLoader(
    val_full_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Train model
print("\nTraining model on full images...")
full_img_results = train_dinov2_classifier(train_full_loader, val_full_loader, fold_idx=999)

# Save full image model
# Save full image model
full_img_model_path = model_save_dir / 'full_image_model.pth'
torch.save(full_img_results['model_state'], full_img_model_path)
print(f"\n Full image model saved: {full_img_model_path}")
print(f"  Best validation accuracy: {full_img_results['best_val_acc']:.2f}%")

In [ ]:
print("="*80)
print("  GENERATING PREDICTIONS FROM FULL IMAGE MODEL")
print("="*80)

test_original_dir = BASE_PATH / 'test_images/mistery_cat'


# Load test images (original, not cropped)
test_full_images = sorted(list(test_original_dir.glob('*.jpg')))
print(f"Found {len(test_full_images)} test images (full)")

# Load the full image model
model_full = DINOv2Classifier(
    backbone=dinov2_backbone,
    num_classes=len(class_mapping),
    hidden_dim=512,
    dropout=0.3
)
model_full.load_state_dict(torch.load(full_img_model_path, map_location=DEVICE))
model_full = model_full.to(DEVICE)
model_full.eval()

# Create test dataset
test_full_dataset = TestDataset(test_full_images, transform=val_transform)
test_full_loader = DataLoader(
    test_full_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Generate predictions
full_img_preds = []
full_img_logits = []

with torch.no_grad():
    for images, names in tqdm(test_full_loader, desc='Full Image Predictions'):
        images = images.to(DEVICE)
        outputs = model_full(images)

        full_img_logits.extend(outputs.cpu().numpy())
        predictions = torch.argmax(outputs, dim=1)
        full_img_preds.extend(predictions.cpu().numpy())

full_img_preds = np.array(full_img_preds)
print(f"\n Generated {len(full_img_preds)} predictions from full image model")

In [ ]:
print("="*80)
print(" FINAL ENSEMBLE: WEIGHTED TTA + FULL IMAGE MODEL")
print("="*80)

# Collect all logits instead of hard predictions
# Re-run predictions to get logits (soft predictions)
print("Collecting logits from all models...")

all_logits = []  # Will store: (n_models, n_samples, n_classes)
model_weights = []  # Validation accuracy weights

# Get logits from each fold with TTA
for fold in range(N_FOLDS):
    print(f"\nFold {fold + 1}/{N_FOLDS}")

    # Load model
    model = DINOv2Classifier(
        backbone=dinov2_backbone,
        num_classes=len(class_mapping),
        hidden_dim=512,
        dropout=0.3
    )
    model_path = model_save_dir / f'fold_{fold}.pth'
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model = model.to(DEVICE)
    model.eval()

    # Weight for this fold (from validation accuracy)
    fold_val_acc = fold_results[fold]['best_val_acc'] / 100.0

    # For each TTA variant
    for tta_idx, tta_transform in enumerate(tta.get_tta_transforms()):
        print(f"  TTA variant {tta_idx + 1}/5", end='\r')

        class TTATestDataset(Dataset):
            def __init__(self, image_paths, transform):
                self.image_paths = image_paths
                self.transform = transform

            def __len__(self):
                return len(self.image_paths)

            def __getitem__(self, idx):
                img_path = self.image_paths[idx]
                image = Image.open(img_path).convert('RGB')
                image = self.transform(image)
                return image, str(img_path.name)

        tta_dataset = TTATestDataset(test_images, tta_transform)
        tta_loader = DataLoader(
            tta_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )

        # Get logits
        batch_logits = []
        with torch.no_grad():
            for images, names in tta_loader:
                images = images.to(DEVICE)
                outputs = model(images)
                batch_logits.append(outputs.cpu().numpy())

        fold_tta_logits = np.vstack(batch_logits)
        all_logits.append(fold_tta_logits)
        model_weights.append(fold_val_acc)

    print(f"  Completed 5 TTA variants (weight: {fold_val_acc:.4f})")

# Get logits from full image model
print("\nFull Image Model")
model_full = DINOv2Classifier(
    backbone=dinov2_backbone,
    num_classes=len(class_mapping),
    hidden_dim=512,
    dropout=0.3
)
model_full.load_state_dict(torch.load(full_img_model_path, map_location=DEVICE))
model_full = model_full.to(DEVICE)
model_full.eval()

test_full_dataset = TestDataset(test_full_images, transform=val_transform)
test_full_loader = DataLoader(
    test_full_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

full_batch_logits = []
with torch.no_grad():
    for images, names in tqdm(test_full_loader, desc='Full Image Logits'):
        images = images.to(DEVICE)
        outputs = model_full(images)
        full_batch_logits.append(outputs.cpu().numpy())

full_img_logits = np.vstack(full_batch_logits)
all_logits.append(full_img_logits)
full_img_weight = full_img_results['best_val_acc'] / 100.0
model_weights.append(full_img_weight)

print(f"  Completed (weight: {full_img_weight:.4f})")

# Convert to arrays
all_logits = np.array(all_logits)  # Shape: (n_models, n_samples, n_classes)
model_weights = np.array(model_weights)  # Shape: (n_models,)

print(f"\nAll logits shape: {all_logits.shape}")
print(f"Model weights shape: {model_weights.shape}")
print(f"Total models: {len(model_weights)}")

# Weighted average of logits (soft voting)
weighted_logits = np.zeros_like(all_logits[0])  # (n_samples, n_classes)

for i, weight in enumerate(model_weights):
    weighted_logits += weight * all_logits[i]

weighted_logits /= model_weights.sum()

# Convert to predictions
ensemble_predictions = np.argmax(weighted_logits, axis=1)

print(f"\n Weighted ensemble predictions: {len(ensemble_predictions)}")

# Create idx to class mapping
idx_to_class = {idx: name for name, idx in class_mapping.items()}

# Show prediction distribution
unique, counts = np.unique(ensemble_predictions, return_counts=True)
print("\nPrediction distribution:")
for cls_idx, count in zip(unique, counts):
    cls_name = idx_to_class[cls_idx]
    print(f"  {cls_name}: {count}")

# Show model weights used
print("\nModel weights:")
for fold in range(N_FOLDS):
    fold_weight = model_weights[fold * 5]  # Same weight for all TTA variants of a fold
    print(f"  Fold {fold + 1} (×5 TTA): {fold_weight:.4f}")
print(f"  Full Image Model: {model_weights[-1]:.4f}")

## Step 12: Detailed Per-Class Metrics for Ensemble

Extract comprehensive per-class performance metrics for the 16-model weighted ensemble

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

def evaluate_ensemble_per_class(all_preds, all_labels, class_names):
    """
    Evaluate per-class metrics for ensemble predictions
    
    Args:
        all_preds: array of predicted labels
        all_labels: array of true labels
        class_names: list of class names
    
    Returns:
        dict with per-class metrics
    """
    # Overall accuracy
    accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
    
    # Classification report
    report = classification_report(all_labels, all_preds, 
                                   target_names=class_names, 
                                   output_dict=True, 
                                   zero_division=0)
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    
    # Per-class accuracies
    per_class_acc = {}
    for i, class_name in enumerate(class_names):
        class_mask = np.array(all_labels) == i
        if class_mask.sum() > 0:
            class_correct = np.sum((np.array(all_preds)[class_mask] == i))
            per_class_acc[class_name] = class_correct / class_mask.sum() * 100
        else:
            per_class_acc[class_name] = 0.0
    
    return {
        'accuracy': accuracy,
        'per_class_accuracy': per_class_acc,
        'classification_report': report,
        'confusion_matrix': cm
    }

print("Per-class evaluation function defined")


### 16-Model Weighted Ensemble Evaluation

This evaluates:
- 3 cropped folds × 5 TTA variants = 15 predictions
- 1 uncropped model = 1 prediction
- Total: 16 weighted predictions per image

In [ ]:
print("="*80)
print("DINOV2 16-MODEL WEIGHTED ENSEMBLE - VALIDATION SET")
print("="*80)

# Get class names
class_names = sorted(list(set([label for _, label in full_dataset])))
print(f"Number of classes: {len(class_names)}")

# Create mapping
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
idx_to_class = {idx: name for name, idx in class_to_idx.items()}

# Prepare dataloader for full dataset
from torch.utils.data import DataLoader

class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, labels, transform):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = class_to_idx[self.labels[idx]]
        return img, label

# Create dataset
image_paths = [img for img, _ in full_dataset]
labels = [label for _, label in full_dataset]
eval_dataset = SimpleDataset(image_paths, labels, val_transform)
eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Evaluation dataset size: {len(eval_dataset)}")


In [ ]:
# Collect predictions from all 16 models
all_predictions = []
all_true_labels = []

# Weights based on validation performance
fold_weights = {
    0: 0.9646,  # Fold 1: 98.23% val accuracy
    1: 0.9367,  # Fold 2: 97.22% val accuracy
    2: 0.9468   # Fold 3: 97.55% val accuracy
}
uncropped_weight = 0.9709  # 97.09% val accuracy, trained on full 1,185 images

print("Loading models and generating predictions...")
print("="*80)

# TTA transforms
tta_transforms = [
    val_transform,  # Original
    transforms.Compose([  # Horizontal flip
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]),
    transforms.Compose([  # Rotate +10
        transforms.Resize((224, 224)),
        transforms.RandomRotation((10, 10)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]),
    transforms.Compose([  # Rotate -10
        transforms.Resize((224, 224)),
        transforms.RandomRotation((-10, -10)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]),
    transforms.Compose([  # Brightness adjustment
        transforms.Resize((224, 224)),
        transforms.ColorJitter(brightness=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
]

# Store all probability predictions
all_probs = []
weights_list = []

# 1. Get predictions from 3 cropped folds with TTA
for fold_idx in range(3):
    print(f"\nProcessing Fold {fold_idx + 1} (cropped)...")
    
    # Load model
    model_path = f'./dinov2_fold_{fold_idx}_best.pth'
    model = DINOv2BirdClassifier(num_classes=len(class_names)).to(device)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    
    # Get predictions with each TTA variant
    for tta_idx, tta_transform in enumerate(tta_transforms):
        print(f"  TTA variant {tta_idx + 1}/5", end='\r')
        
        # Create dataloader with this TTA transform
        tta_dataset = SimpleDataset(image_paths, labels, tta_transform)
        tta_loader = DataLoader(tta_dataset, batch_size=32, shuffle=False, num_workers=2)
        
        fold_probs = []
        fold_labels = []
        
        with torch.no_grad():
            for images, batch_labels in tta_loader:
                images = images.to(device)
                outputs = model(images)
                probs = torch.nn.functional.softmax(outputs, dim=1)
                fold_probs.append(probs.cpu().numpy())
                fold_labels.extend(batch_labels.numpy())
        
        fold_probs = np.vstack(fold_probs)
        all_probs.append(fold_probs)
        weights_list.append(fold_weights[fold_idx])
        
        if tta_idx == 0:  # Store labels only once per fold
            all_true_labels = fold_labels
    
    print(f"  Fold {fold_idx + 1} completed (5 TTA variants)")

# 2. Get predictions from uncropped model
print(f"\nProcessing Uncropped Model...")
model_path = './dinov2_uncropped_best.pth'
model = DINOv2BirdClassifier(num_classes=len(class_names)).to(device)
model.load_state_dict(torch.load(model_path))
model.eval()

uncropped_probs = []
with torch.no_grad():
    for images, batch_labels in eval_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        uncropped_probs.append(probs.cpu().numpy())

uncropped_probs = np.vstack(uncropped_probs)
all_probs.append(uncropped_probs)
weights_list.append(uncropped_weight)

print(f"Uncropped model completed")

print(f"\nTotal predictions collected: {len(all_probs)}")
print(f"Total weights: {len(weights_list)}")


In [ ]:
# Weighted average of probabilities
print("\nComputing weighted ensemble predictions...")
weights_array = np.array(weights_list).reshape(-1, 1, 1)  # Shape: (16, 1, 1)
all_probs_array = np.array(all_probs)  # Shape: (16, n_samples, n_classes)

# Weighted average: sum(w_i * P_i) / sum(w_i)
weighted_probs = np.sum(all_probs_array * weights_array, axis=0) / np.sum(weights_list)

# Get final predictions
ensemble_preds = np.argmax(weighted_probs, axis=1)

# Evaluate metrics
metrics = evaluate_ensemble_per_class(ensemble_preds, all_true_labels, class_names)

print("="*80)
print("DINOV2 16-MODEL WEIGHTED ENSEMBLE - RESULTS")
print("="*80)
print(f"\nOverall Validation Accuracy: {metrics['accuracy']*100:.2f}%\n")

print(f"{'Class':<30} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
print("-"*80)

report = metrics['classification_report']
for class_name in sorted(class_names):
    if class_name in report:
        acc = metrics['per_class_accuracy'][class_name]
        prec = report[class_name]['precision'] * 100
        rec = report[class_name]['recall'] * 100
        f1 = report[class_name]['f1-score'] * 100
        print(f"{class_name:<30} {acc:>6.1f}%      {prec:>6.1f}%      {rec:>6.1f}%      {f1:>6.1f}%")

print("\n" + "="*80)


### Save Per-Class Metrics to CSV

In [ ]:
# Compile per-class metrics into CSV
print("Saving per-class metrics to CSV...")

rows = []
report = metrics['classification_report']

for class_name in sorted(class_names):
    if class_name in report:
        row = {
            'Class': class_name,
            'Accuracy': f"{metrics['per_class_accuracy'][class_name]:.1f}",
            'Precision': f"{report[class_name]['precision']*100:.1f}",
            'Recall': f"{report[class_name]['recall']*100:.1f}",
            'F1_Score': f"{report[class_name]['f1-score']*100:.1f}",
            'Support': int(report[class_name]['support'])
        }
        rows.append(row)

df_per_class = pd.DataFrame(rows)

# Save to CSV
output_path = './dinov2_ensemble_per_class_metrics.csv'
df_per_class.to_csv(output_path, index=False)
print(f"\nPer-class metrics saved to: {output_path}")
print("\n" + df_per_class.to_string(index=False))

# Also save summary
summary = {
    'Model': 'DINOv2 16-Model Weighted Ensemble',
    'Configuration': '3 cropped folds × 5 TTA + 1 uncropped',
    'Total_Models': 16,
    'Validation_Accuracy': f"{metrics['accuracy']*100:.2f}",
    'Macro_Avg_Precision': f"{report['macro avg']['precision']*100:.2f}",
    'Macro_Avg_Recall': f"{report['macro avg']['recall']*100:.2f}",
    'Macro_Avg_F1': f"{report['macro avg']['f1-score']*100:.2f}"
}

df_summary = pd.DataFrame([summary])
summary_path = './dinov2_ensemble_summary.csv'
df_summary.to_csv(summary_path, index=False)
print(f"\nSummary saved to: {summary_path}")
print("\n" + df_summary.to_string(index=False))
